# Cuaderno asociado al TFG

Este cuaderno forma parte del repositorio asociado al Trabajo de Fin de Grado **“Impacto de las Telecomunicaciones en la Agricultura 4.0”**.

Por motivos de confidencialidad, los archivos de datos reales no se incluyen en el repositorio. El código se mantiene como referencia metodológica y está preparado para trabajar con archivos Excel equivalentes ubicados en la carpeta `Datos/`.


In [ ]:
#!pip install numpy-financial


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [ ]:
import numpy_financial as npf


In [ ]:
import matplotlib.pyplot as plt

Se han utilizado precios medios para todos los insumios así como precio medio de venta por tonelada de cada año o precio de secadero igual para todos los años

Esto permite aislar el impacto estructural de la digitalización sobre la eficiencia técnico económica, evitando que la volatilidad del mercado contamine la estimación. 


In [ ]:

df = pd.read_excel("../Datos/Balance_economico.xlsx")

df.columns = df.columns.str.strip()
df["Cultivo"] = df["Cultivo"].str.strip()
df["Parcela"] = df["Parcela"].astype(str).str.strip()

# Crear índice tecnológico
tech_cols = ["GPS", "RTK", "ISOBUS", "Siembra_variable", "Abono_variable", "Corte_tramos"]
df[tech_cols] = df[tech_cols].fillna(0).astype(int)
df["tech_index"] = df[tech_cols].sum(axis=1)


In [ ]:
#separación por cultivo
df_maiz = df[df["Cultivo"] == "Maiz"].copy()
df_arroz = df[df["Cultivo"] == "Arroz"].copy()
df_tomate = df[df["Cultivo"] == "Tomates"].copy()


In [ ]:
#modelo base OLS

formula_base = """
Beneficio_por_ha ~ tech_index
                  + Humedad
                  + Superficie
                  + C(Año)
"""

m_maiz_base = smf.ols(formula_base, data=df_maiz).fit(cov_type="HC3")
m_arroz_base = smf.ols(formula_base, data=df_arroz).fit(cov_type="HC3")

print("MAÍZ - BASE")
print(m_maiz_base.summary())

print("\nARROZ - BASE")
print(m_arroz_base.summary())


MAIZ
muestra un efecto positivo económicamente relevante: +265,45€/t con evidencia estadística moderada, alpha =0,1,. 
p valor 0,069 significativo
R^2 0,860 el modelo explica el 86% de la variabilidad del beneficio
La variabilidad del beneficio parece estar más vinculada al nivel tecnológico que a sucesos anuales 

Arroz: 
tech index = +131 €/t por punto de tech index. su p valor es muy bajo, por lo que el efecto es muy robusto
Explotaciones mayores obtienen beneficio algo mayor por hectárea: superficie = +4€/ha


In [ ]:
#efectos fijos por parcela
#comparamos las parcelas consigo misma a lo largo del tiempo
formula_fe = """
Beneficio_por_ha ~ tech_index + C(Año) + C(Parcela)
"""

m_maiz_fe = smf.ols(formula_fe, data=df_maiz).fit(cov_type="HC3")
m_arroz_fe = smf.ols(formula_fe, data=df_arroz).fit(cov_type="HC3")

print("MAÍZ - FE Parcela + Año")
print(m_maiz_fe.summary())

print("\nARROZ - FE Parcela + Año")
print(m_arroz_fe.summary())


Modelo con efectos fijos por parcela + año

MAIZ:
tech_index = 38,86 €/ha, menor aumento aunque sigue siendo claro el mayor beneficio

ARROZ: 
tech_index = 130€/ha, aún mayor aumento real dentro de cada parcela

Dentro de la misma parcela y controlado por el año, cuando aumenta el nivel tecnológico, el beneficio aumenta aproximadamente la cifra dada


También hay que tener en cuenta que no todos los años en todas las parcelas se han utilizado todas las tecnologías ya que muchas veces por falta de tiempo es imposible de hacer todo con las mismas máquinas y hay que llevarlas acabo con otras


FE= fixed effects= efectos fijos
C(año) : cad año tiene su propio nivel base
C(Parcela): cada parcela tiene su propio nivel base

In [ ]:
#que tecnología aporta más valor
tech_formula = """
Beneficio_por_ha ~ GPS + RTK + ISOBUS + Siembra_variable + Abono_variable + Corte_tramos
                  + Precio_tonelada_venta
                  + Precio_litro_gasoleoB
                  + Humedad
                  + C(Año)
"""

modelo_tech_maiz = smf.ols(tech_formula, data=df_maiz).fit(cov_type="HC3")
print("\nMAIZ - Tecnologías individuales (coef, pval)")
print(pd.DataFrame({
    "coef": modelo_tech_maiz.params,
    "pval": modelo_tech_maiz.pvalues
}).sort_values("pval"))


modelo_tech_arroz = smf.ols(tech_formula, data=df_arroz).fit(cov_type="HC3")
print("\nARROZ - Tecnologías individuales (coef, pval)")
print(pd.DataFrame({
    "coef": modelo_tech_arroz.params,
    "pval": modelo_tech_arroz.pvalues
}).sort_values("pval"))



Ahora vamos a ver que tecnología concreta aporta mayor valor económico

MAIZ: 
la tecnología que realmente parece aportar valor económico es la siembra varaible
Y en menor medida, el corte por tramos
siembre variable: 
+185 €/ha
p = 0.022
→ Significativa al 5%
corte a tramos:
+382 €/ha
p = 0.096
→ Significativa al 10%


ARROZ
practicamente todas las nuevas tecnologías aportan un valor significativo 
1️⃣ Siembra variable (~163 €/ha)
2️⃣ Corte de tramos (~165 €/ha)

aunque los p valores son bastante altos por lo que se observan problemas de colinealidad, que impiden identificar de forma robusta el impacto marginal de cada herramienta por separado

In [ ]:
formula_scale = """
Beneficio_por_ha ~ tech_index * Superficie + Humedad + C(Año)
"""

m_maiz_scale = smf.ols(formula_scale, data=df_maiz).fit(cov_type="HC3")
m_arroz_scale = smf.ols(formula_scale, data=df_arroz).fit(cov_type="HC3")

print("MAÍZ - escala")
print(m_maiz_scale.summary())

print("\nARROZ - escala")
print(m_arroz_scale.summary())


Vemos si la digitalización es más rentable en explotaciones grandes

MAIZ: el efecto medio de la tecnología sigue siendo parecido al modelo base: 267 €/ha por lo que la interacción con superficie no es significativa

ARROZ: de la misma manera en el arroz, el efecto tecnológico sigue siendo fuerte pero muy parecido con respecto al tamaño de la superficie

In [ ]:
#conexión tecnico economico
formula_bridge = """
Beneficio_por_ha ~ tech_index
                  + Rend_t_ha
                  + Consumo_L_ha
                  + Productividad_ha_h
                  + Humedad
                  + Superficie
                  + C(Año)
"""

m_maiz_bridge = smf.ols(formula_bridge, data=df_maiz).fit(cov_type="HC3")
m_arroz_bridge = smf.ols(formula_bridge, data=df_arroz).fit(cov_type="HC3")

print("MAÍZ - bridge")
print(m_maiz_bridge.summary())

print("\nARROZ - bridge")
print(m_arroz_bridge.summary())


En maiz, el impacto económico de la digitalización se explica principalemten a través del incremento en el rendimiento productivo, de donde viene el aumento del beneficio económico principal sobre este

En arroz el patrón es más completo, tienen importancia tanto el incremento del rendimiento productivo como la reducción del consumo energético y la mejora de la productividad. 
ahora tech_index pasa a ser 99, por lo que aproximadamente 30€/ha del efecto tecnológico se explican por mejorar en rendimiento, consumo y productivididad.


In [ ]:
#elasticidad económica
df_maiz["Beneficio_log"] = np.log(df_maiz["Beneficio_por_ha"])

m_log = smf.ols("Beneficio_log ~ tech_index + C(Año)", data=df_maiz).fit(cov_type="HC3")
print(m_log.summary())

por cada punto en tech_index, se interpreta como +11,7% de incremento de beneficio por hectárea

In [ ]:
df_maiz["Grupo_tech"] = np.where(df_maiz["tech_index"] >= df_maiz["tech_index"].median(), "Alto", "Bajo")
df_maiz.groupby("Grupo_tech")[["Beneficio_por_ha",
                                "Rend_t_ha",
                                "Consumo_L_ha"]].mean()


Interpretación económica clara con alta tecnología

beneficio: +727€/ha

alta tecnología:+2,67t/ha, mayor rendimienot

consumo: -1,85L/ha, mejor eficiencia energética

In [ ]:
#elasticidad económica arroz
df_arroz["Beneficio_log"] = np.log(df_arroz["Beneficio_por_ha"])

m_log_arroz = smf.ols(
    "Beneficio_log ~ tech_index + C(Año)",
    data=df_arroz
).fit(cov_type="HC3")

print("ARROZ - Elasticidad (log)")
print(m_log_arroz.summary())

no podemos sacar muchas conclusiones ya que tiene una p muy alta, implicando que es poco significativo

Por lo que este modelo log para arroz no es fiable debido al tamaño muestral extremadamente reducido

In [ ]:
#comparacion alto vs bajo tech
df_arroz["Grupo_tech"] = np.where(df_arroz["tech_index"] >= df_arroz["tech_index"].median(), "Alto", "Bajo")
df_arroz.groupby("Grupo_tech")[["Beneficio_por_ha",
                                "Rend_t_ha",
                                "Consumo_L_ha"]].mean()

Hay una diferencia enorme de +496€/ha de manera que las explotaciones con mayor nivel tecnológico pasan de pérdidas importantes a comenzar a tener beneficio positivo

En cuanto al rendimiento, es de +1,1 toneladas por hectárea, la tecnología está asociada a mayor producción física. siendo un cultivo con una media de aproximadamente 8 toneladas por hectárea de producción, dicho aumento es un gran avance significativo

consumo energético: -3,2 litros por hectárea, reduciendo el consumo de gasoil

In [ ]:
#descomposicion del efecto tecnológico
formula_total = """
Beneficio_por_ha ~ tech_index + Humedad + Superficie + C(Año)
"""

formula_bridge = """
Beneficio_por_ha ~ tech_index
                + Rend_t_ha + Consumo_L_ha + Productividad_ha_h
                + Humedad + Superficie + C(Año)
"""

def decomposition(df_crop, crop_name):
    # Modelo total
    m_total = smf.ols(formula_total, data=df_crop).fit(cov_type="HC3")
    beta_total = m_total.params["tech_index"]
    
    # Modelo bridge
    m_bridge = smf.ols(formula_bridge, data=df_crop).fit(cov_type="HC3")
    beta_bridge = m_bridge.params["tech_index"]
    
    # Descomposición
    explained = beta_total - beta_bridge
    explained_pct = explained / beta_total * 100 if beta_total != 0 else np.nan
    remaining_pct = beta_bridge / beta_total * 100 if beta_total != 0 else np.nan
    
    out = {
        "Cultivo": crop_name,
        "Efecto_total_tech (€/ha por punto)": beta_total,
        "Efecto_directo_residual (€/ha por punto)": beta_bridge,
        "Parte_explicada_por_canales (€/ha)": explained,
        "%_explicado_por_canales": explained_pct,
        "%_residual_directo": remaining_pct,
        "p_total": m_total.pvalues["tech_index"],
        "p_bridge": m_bridge.pvalues["tech_index"],
        "N": int(m_total.nobs)
    }
    return out, m_total, m_bridge

# Ejecutar para MAÍZ y ARROZ
res_maiz, m_total_maiz, m_bridge_maiz = decomposition(df_maiz, "Maíz")
res_arroz, m_total_arroz, m_bridge_arroz = decomposition(df_arroz, "Arroz")

tabla_descomp = pd.DataFrame([res_maiz, res_arroz])
print(tabla_descomp)

MAIZ:
la digitalización aumenta el beneficio de forma económicamente relevante. 
Modelo total: +265,45€/ha con p=0,069 significativo
POr lo que no hay efecto independiente adicional, practicamente todo el impacto de la digitalización se transmite a través del aumento del rendimiento productivo

ARROZ: 
efecto tech index +131 €/ha
efecto residual: 98,9 €/ha, con p=0,0018
parte explicada por canales técnicos: 32,11€/ha, %explicado = 24,5%, parte del efecto que viene por mejoras técnicas
Por lo que en el arroz la introducción de las tecnologías tiene un efecto económico más amplio: mejor gestión, planificacion, rediccion de insumos. 


In [ ]:
print("\n--- MAÍZ: resumen modelo total ---")
print(m_total_maiz.summary())

R^2 =0,86
tech index = +265,45€/ha. 
por lo que cada punto adicional de digitalización aumenta el beneficio en 265 €/ha
Es muy relevante económicamente. 
Existe una relación positiva clara entre digitalización y beneficio 

In [ ]:
print("\n--- MAÍZ: resumen modelo bridge ---")
print(m_bridge_maiz.summary())

R^2 sube a 0,918
La digitalización aumenta el beneficio principalmente porque aumenta el rendimiento económico
practicamente todo el efecto tecnológico es a través de esto.

In [ ]:
print("\n--- ARROZ: resumen modelo total ---")
print(m_total_arroz.summary())

cada punto adicional de digitalización aumenta el beneficio en aproximadamente 131 €/ha, con gran evidencia estadística. El efecto tecnológico es claramente el principal determinante, mientras que humedad o superficie no son significativas. 


In [ ]:
print("\n--- ARROZ: resumen modelo bridge ---")
print(m_bridge_arroz.summary())

tech_index: 98,9 €/ha, con p=0,002
rendimiento significativo, consumo significativo y productividad igual


In [ ]:
#impacto relativo, % sobre el beneficio medio
def relative_impact(df_crop, crop_name, beta_total):
    mean_benef = df_crop["Beneficio_por_ha"].mean()
    median_benef = df_crop["Beneficio_por_ha"].median()
    
    rel_mean = beta_total / mean_benef * 100 if mean_benef != 0 else np.nan
    rel_median = beta_total / median_benef * 100 if median_benef != 0 else np.nan
    
    return {
        "Cultivo": crop_name,
        "Beneficio_medio (€/ha)": mean_benef,
        "Beneficio_mediana (€/ha)": median_benef,
        "Impacto_tech_total (€/ha por punto)": beta_total,
        "%_sobre_media": rel_mean,
        "%_sobre_mediana": rel_median
    }

rel_maiz = relative_impact(df_maiz, "Maíz", res_maiz["Efecto_total_tech (€/ha por punto)"])
rel_arroz = relative_impact(df_arroz, "Arroz", res_arroz["Efecto_total_tech (€/ha por punto)"])

tabla_rel = pd.DataFrame([rel_maiz, rel_arroz])
print(tabla_rel)


impacto relativo

Maiz: 
Beneficio medio: 1.341 €/ha
Beneficio mediana: 1.437 €/ha
Impacto tech total: +265 €/ha
Impacto relativo:
≈ 19,8% sobre la media
≈ 18,5% sobre la mediana

Un punto adicional de digitalización supone aproximadamente un 19% del beneficio medio anual por hectárea.


ARROZ: 
Beneficio medio: –49,08 €/ha
Beneficio mediana: 23,41 €/ha
Impacto tech total: +131 €/ha

Los porcentajes salen desorvitados ya que en un principio los rendimientos de este cultivo eran negativos, pero al introducir estas tecnologías se ha vuelto positivo el rendimiento.
Dado el reducido margen medio en arroz, el impacto absoluto de la digitalización representa una mejora económica sustancial que puede alterar significativamente la rentabilidad del cultivo.

In [ ]:
# Rendimiento medio por año - MAÍZ
rend_maiz_anual = (
    df_maiz
    .groupby("Año")["Rend_t_ha"]
    .mean()
    .reset_index()
    .rename(columns={"Rend_t_ha": "Rendimiento_medio_t_ha"})
)

print("MAÍZ - Rendimiento medio por año (t/ha)")
print(rend_maiz_anual)


In [ ]:
# Rendimiento medio por año - ARROZ
rend_arroz_anual = (
    df_arroz
    .groupby("Año")["Rend_t_ha"]
    .mean()
    .reset_index()
    .rename(columns={"Rend_t_ha": "Rendimiento_medio_t_ha"})
)

print("\nARROZ - Rendimiento medio por año (t/ha)")
print(rend_arroz_anual)


In [ ]:
# Unimos por Año
tabla_rendimientos = pd.concat([rend_maiz_anual, rend_arroz_anual], axis=1)

print("Rendimiento medio por año (t/ha)")
print(tabla_rendimientos.round(3))


En el maiz, entre 2020-2021 hay 3 toneladas de diferencia por hectarea. 
Más allá de que el 2020 no fue el mejor año en cuanto a producciones, si que resulta llamativo que el año que empezamos a utilizar más todos estos avances, se ha conseguir conseguir un +23% del aumento de producción y que se ha conseguido mantener de manera más o menos constante durante los siguientes años

De manera parecida en el arroz, ha habido un aumento de 1,19t/ha lo que ha sido de entorno al 16% y que también se ha conseguido mantener de manera uniforme por los siguientes años 


In [ ]:
#Gasto medio por hectárea cada año
# MAÍZ
coste_maiz_anual = (
    df_maiz
    .groupby("Año")["Coste_total_por_ha"]
    .mean()
    .rename("Maíz_coste_€/ha")
)

# ARROZ
coste_arroz_anual = (
    df_arroz
    .groupby("Año")["Coste_total_por_ha"]
    .mean()
    .rename("Arroz_coste_€/ha")
)

# Tabla comparativa
tabla_costes = pd.concat([coste_maiz_anual, coste_arroz_anual], axis=1)

print("Coste medio por hectárea y año (€)")
print(tabla_costes.round(2))


De igual manera que antes, vemos un descenso considerable del año 2020 al año 2021, llegando hasta el año 2025 donde ha sido el nivel más bajo de todos ellos
se ha conseguido una rediccion de entorno a 90 €/ha desde 2020
Aunque el principal beneficio en el maiz proviene del aumento de la producción.

de manera más acentuada lo podemos obtener en el arroz, con una bajada de 203€/ha, manteniendo más o menos dicho nivel por lo siguientes años. 
la principal mejora en el arroz proviene de la reducción de los costes de los insumios.

El análisis descriptivo de costes muestra una reducción significativa en el gasto medio por hectárea a partir de 2021, coincidiendo con la intensificación del uso tecnológico. En arroz, la caída de costes entre 2020 y 2021 supera los 200 €/ha, lo que sugiere una mejora sustancial en la eficiencia productiva. En maíz, la reducción es más moderada, siendo el principal mecanismo de mejora económica el incremento del rendimiento.

In [ ]:
#variación interanual
tabla_costes_var = tabla_costes.pct_change() * 100
print("Variación % interanual del coste medio")
print(tabla_costes_var.round(2))


en 2021 observamos de nuevo una clara reducción, y aunque luego haya habido fluctuaciones pequeñas ya que no todos los años hemos podido utilziar todas las tecnologías en todas las parcelas, la tendencia es claramente descendente en el gasto de insumios

El mayor descenso se ha producido en el arroz entre el año 2020 y 2021 con cas 8,5% de descenso del coste de insumios

TOMATE

In [ ]:
# MODELO BASE TOMATE
m_total_tomate = smf.ols(
    "Beneficio_por_ha ~ tech_index + C(Año)",
    data=df_tomate
).fit(cov_type="HC3")

print("TOMATE - Modelo base")
print(m_total_tomate.summary())


In [ ]:
# Impacto relativo tomate
mean_benef_tomate = df_tomate["Beneficio_por_ha"].mean()

beta_tomate = m_total_tomate.params["tech_index"]

impacto_rel_tomate = beta_tomate / mean_benef_tomate * 100

print("Impacto tech tomate (€/ha):", round(beta_tomate,2))
print("% sobre beneficio medio:", round(impacto_rel_tomate,2))


In [ ]:
rend_tomate_anual = (
    df_tomate
    .groupby("Año")["Rend_t_ha"]
    .mean()
)

print("Tomate - Rendimiento medio por año")
print(rend_tomate_anual.round(2))


En el cultivo de tomate, el análisis exploratorio sugiere un posible efecto positivo de la digitalización sobre el beneficio por hectárea (+752 €/ha por punto de índice tecnológico). Sin embargo, el reducido tamaño muestral (N=5) impide obtener evidencia estadística robusta, por lo que estos resultados deben interpretarse con cautela y carácter meramente indicativo.

IMPLANTACIÓN TECNOLOGÍA

In [ ]:
# COSTES DE IMPLANTACIÓN (€)
# ==============================

costes_tec = pd.DataFrame({
    "Tecnologia": [
        "GPS",
        "RTK_incremental",
        "ISOBUS",
        "Siembra_variable",
        "Abono_variable",
        "Corte_tramos"   # Máquina de fitosanitarios
    ],
    "Coste_eur": [
        11000,   # GPS
        7000,    # RTK adicional sobre GPS (18.000 - 11.000)
        2500,    # ISOBUS
        35000,   # Sembradora variable
        25000,   # Abonadora variable
        62000    # Máquina fitosanitarios (Corte_tramos)
    ]
})

In [ ]:
#impacto de cada tecnología sobre beneficio por ha
def calcular_roi_tecnologias(df_crop, crop_name, superficies=(50, 100, 200)):

    df = df_crop.copy()

    # RTK incremental
    if "RTK" in df.columns:
        df["RTK_incremental"] = df["RTK"]

    # Tecnologías disponibles
    tech_vars = [
        "GPS",
        "RTK_incremental",
        "ISOBUS",
        "Siembra_variable",
        "Abono_variable",
        "Corte_tramos"
    ]

    tech_vars = [t for t in tech_vars if t in df.columns]

    # Controles si existen
    controles = []
    for c in ["Humedad", "Superficie"]:
        if c in df.columns:
            controles.append(c)

    # Efectos fijos por año si existe
    fe = ["C(Año)"] if "Año" in df.columns else []

    formula = "Beneficio_por_ha ~ " + " + ".join(tech_vars + controles + fe)

    modelo = smf.ols(formula, data=df).fit(cov_type="HC3")

    # Crear tabla base
    tabla = pd.DataFrame({
        "Cultivo": crop_name,
        "Tecnologia": tech_vars,
        "Efecto_€/ha": [modelo.params.get(v, np.nan) for v in tech_vars],
        "p_valor": [modelo.pvalues.get(v, np.nan) for v in tech_vars],
        "N": int(modelo.nobs)
    })

    # Unir con costes
    tabla = tabla.merge(costes_tec, on="Tecnologia", how="left")

    # Calcular métricas para distintos tamaños
    for ha in superficies:
        tabla[f"Beneficio_anual_{ha}ha_€"] = tabla["Efecto_€/ha"] * ha
        tabla[f"Payback_{ha}ha_años"] = tabla["Coste_eur"] / tabla[f"Beneficio_anual_{ha}ha_€"]
        tabla[f"ROI_{ha}ha_%"] = (tabla[f"Beneficio_anual_{ha}ha_€"] / tabla["Coste_eur"]) * 100

    # Ranking según 100 ha (puedes cambiar)
    if f"Payback_{superficies[1]}ha_años" in tabla.columns:
        tabla = tabla.sort_values(f"Payback_{superficies[1]}ha_años")

    return tabla, modelo


In [ ]:
# ==============================
# MAÍZ
# ==============================

tabla_maiz, modelo_maiz = calcular_roi_tecnologias(df_maiz, "Maíz")

print("\n===== MAÍZ: ROI y Payback =====")
print(tabla_maiz.round(2))


# ==============================
# ARROZ
# ==============================

tabla_arroz, modelo_arroz = calcular_roi_tecnologias(df_arroz, "Arroz")

print("\n===== ARROZ: ROI y Payback =====")
print(tabla_arroz.round(2))


MAIZ

ISOBUS: +168€/ha, con 100 hectáreas, el payback sería en 0,15 años
ROI 100 ha: 674% anual
Es la tecnología ocn mejor rentabilidad relativa. coste bajo y efecto positivo, amotización practicamente inmediata
RTK: +168€/ha, payback en 0,42 años, 241% anual. muy rentable

siembra variable: 
+185€/ha, payback en 1,88 años, 53% anual. es la tecnología estructural más sólida en maíz. 

Corte tramos: +382/ha. payback 1,62 años. ROI 62%
a pesar de ser la inversión más alta, genera un impacto muy elevado. se puede amortizar en menos de dos años con 100 ha

Abono variable. +14€/ha, ROI bajo, payback largo. no parece generar mejoras claras en beneficio neto en maiz


ARROZ: 
aunque se observa p valor alto, lo que implica multicolinealidad, pero observandolo desde el punto económico:
Tanto la siembra variable como el corte por tramos son las tecnologías que mayor impacto tienen sobre la producción. Sobretodo tienen un gran impacto en el ahorro del gasto de insumos. 



In [ ]:
#VAN: valor actual neto
#TIR: tasa interna de retorno
#Tasa de descuento usada del 5%

def calcular_van_tir(tabla_roi, superficie=100, tasa_descuento=0.05, vida_util=10):

    resultados = tabla_roi.copy()

    van_list = []
    tir_list = []

    for _, row in resultados.iterrows():

        beneficio_anual = row[f"Beneficio_anual_{superficie}ha_€"]
        inversion = row["Coste_eur"]

        flujos = [-inversion] + [beneficio_anual] * vida_util

        # VAN
        van = npf.npv(tasa_descuento, flujos)

        # TIR
        try:
            tir = npf.irr(flujos)
        except:
            tir = np.nan

        van_list.append(van)
        tir_list.append(tir)

    resultados[f"VAN_{superficie}ha_€"] = van_list
    resultados[f"TIR_{superficie}ha_%"] = np.array(tir_list) * 100

    resultados = resultados.sort_values(f"VAN_{superficie}ha_€", ascending=False)

    return resultados


In [ ]:
#para maiz en 100 ha
maiz_van = calcular_van_tir(tabla_maiz, superficie=100)

print("\n==== MAÍZ - VAN y TIR (100 ha) ====")
print(maiz_van.round(2))


para maiz: 
1. ISOBUS → Inversión estrella

VAN ≈ 127.715 €

TIR ≈ 674%

Payback ≈ 0,15 años
Es la tecnología más eficiente económicamente en maíz.
Coste muy bajo y efecto elevado → creación de valor extraordinaria.

2. RTK incremental

VAN ≈ 123.215 €

TIR ≈ 241%

Payback ≈ 0,42 años

 Conclusión:

Muy rentable, especialmente cuando se complementa con GPS.
Se amortiza prácticamente en el primer año.

3. Corte_tramos (máquina fitosanitarios)

VAN ≈ 233.682 € (el más alto en valor absoluto)

TIR ≈ 61%

Payback ≈ 1,62 años

 Conclusión clave:

Aunque es la inversión más cara, genera el mayor valor económico acumulado.
Es estratégica para explotaciones medias-grandes.

Siembra variable

VAN ≈ 108.459 €

TIR ≈ 52%

Payback ≈ 1,88 años

 Conclusión:

Es una inversión sólida, con rentabilidad clara y robusta estadísticamente.

Abono variable

VAN ≈ -13.537 €

TIR ≈ -8,5%

Payback muy largo

 Conclusión:

No crea valor económico en el escenario analizado.

GPS solo

VAN ≈ -77.935 €

No tiene TIR

Impacto negativo

 Conclusión:

El GPS por sí solo no es una inversión rentable en maíz.
Su valor aparece cuando se combina con RTK.

In [ ]:
arroz_van = calcular_van_tir(tabla_arroz, superficie=100)

print("\n==== ARROZ - VAN y TIR (100 ha) ====")
print(arroz_van.round(2))


1. Siembra variable

VAN ≈ 91.530 €

TIR ≈ 45,7%

Payback ≈ 2,14 años

 Conclusión:

Es la tecnología más atractiva en arroz.sobretodo a raiz del ahorro que permite dado el alto valor de las semillas de arroz

2. Corte_tramos

VAN ≈ 65.669 €

TIR ≈ 23,4%

Payback ≈ 3,75 años

 Conclusión:

Rentable, pero necesita tamaño suficiente para justificarse.

3. ISOBUS

VAN ≈ 43.189 €

TIR ≈ 236%

Payback ≈ 0,42 años

 Muy interesante:
Aunque el efecto €/ha es menor que en maíz, el bajo coste lo hace muy atractivo.

RTK incremental

VAN ≈ 38.689 €

TIR ≈ 84%

Payback ≈ 1,18 años

Abono variable

VAN ≈ 18.826 €

TIR ≈ 18%

Rentable pero moderadamente.

CONCLUSIONES

La digitalización crea valor económico real

En ambos cultivos, varias tecnologías presentan:

VAN positivo

TIR muy superior al 5%

Periodos de recuperación cortos


El tamaño de explotación es determinante

A 50 ha algunas inversiones ya son rentables

A 100 ha casi todas (salvo GPS solo) generan valor

A 200 ha la rentabilidad se dispara

Esto conecta perfectamente con tu modelo de escala.


Las tecnologías de bajo coste tienen mayor eficiencia relativa

ISOBUS y RTK muestran:

TIR extremadamente altas

Recuperación inmediata

Las inversiones estructurales generan mayor VAN absoluto

La máquina de fitosanitarios (Corte_tramos):

Mayor creación de valor total

Rentabilidad sólida

Inversión estratégica


Los resultados muestran que la digitalización agrícola no solo incrementa el beneficio por hectárea, sino que constituye una inversión financieramente sólida. Tecnologías como ISOBUS y RTK presentan tasas internas de retorno excepcionalmente elevadas, mientras que inversiones estructurales como la maquinaria de aplicación de fitosanitarios generan el mayor valor acumulado en explotaciones medias y grandes. El tamaño de la explotación emerge como factor clave en la viabilidad económica de la adopción tecnológica.

In [ ]:
# HECTÁREAS MÍNIMAS PARA QUE SEA RENTABLE
# ==========================================================

def calcular_hectareas_minimas(tabla_roi):

    tabla = tabla_roi.copy()

    hectareas_min = []

    for _, row in tabla.iterrows():

        efecto = row["Efecto_€/ha"]
        coste = row["Coste_eur"]

        if efecto > 0:
            h_min = coste / efecto
        else:
            h_min = np.nan

        hectareas_min.append(h_min)

    tabla["Hectareas_minimas_rentables"] = hectareas_min

    return tabla[["Tecnologia", "Efecto_€/ha", "Coste_eur", "Hectareas_minimas_rentables"]].sort_values("Hectareas_minimas_rentables")


In [ ]:
print("\n=== MAÍZ - Hectáreas mínimas necesarias ===")
print(calcular_hectareas_minimas(tabla_maiz).round(2))

In [ ]:
print("\n=== ARROZ - Hectáreas mínimas necesarias ===")
print(calcular_hectareas_minimas(tabla_arroz).round(2))

In [ ]:
#simular escenario realista con vida util 12-15 años

def van_tir_por_vida(tabla_roi, superficie=100, tasa_descuento=0.05, vidas=(13,14,15)):
    """
    tabla_roi debe tener:
      - Tecnologia
      - Coste_eur
      - Efecto_€/ha
      - Beneficio_anual_{superficie}ha_€
    """
    out = tabla_roi.copy()

    # Hectáreas mínimas rentables (umbral)
    out["Hectareas_minimas_rentables"] = np.where(
        out["Efecto_€/ha"] > 0, out["Coste_eur"] / out["Efecto_€/ha"], np.nan
    )

    for vida in vidas:
        vans = []
        tirs = []
        for _, r in out.iterrows():
            inv = r["Coste_eur"]
            b_anual = r[f"Beneficio_anual_{superficie}ha_€"]
            flujos = [-inv] + [b_anual] * vida

            van = npf.npv(tasa_descuento, flujos)
            try:
                tir = npf.irr(flujos)
            except Exception:
                tir = np.nan

            vans.append(van)
            tirs.append(tir * 100 if pd.notnull(tir) else np.nan)

        out[f"VAN_{superficie}ha_{vida}a_€"] = vans
        out[f"TIR_{superficie}ha_{vida}a_%"] = tirs

    # Orden útil: por VAN a 10 años (o última vida)
    out = out.sort_values(f"VAN_{superficie}ha_{vidas[-1]}a_€", ascending=False)

    cols_base = ["Tecnologia", "Efecto_€/ha", "Coste_eur", "Hectareas_minimas_rentables"]
    cols_van = [f"VAN_{superficie}ha_{v}a_€" for v in vidas]
    cols_tir = [f"TIR_{superficie}ha_{v}a_%" for v in vidas]

    return out[cols_base + cols_van + cols_tir]

# ====== EJECUCIÓN (elige superficie que te interese) ======
superficie = 100
tasa = 0.05
vidas = (13, 14, 15)

tabla_maiz_vida = van_tir_por_vida(tabla_maiz, superficie=superficie, tasa_descuento=tasa, vidas=vidas)
tabla_arroz_vida = van_tir_por_vida(tabla_arroz, superficie=superficie, tasa_descuento=tasa, vidas=vidas)

print(f"\n=== MAÍZ | Sensibilidad vida útil {vidas} años | {superficie} ha | r={tasa*100:.1f}% ===")
print(tabla_maiz_vida.round(2))

print(f"\n=== ARROZ | Sensibilidad vida útil {vidas} años | {superficie} ha | r={tasa*100:.1f}% ===")
print(tabla_arroz_vida.round(2))


maiz

ISOBUS

VAN (15 años): 172.537 €

TIR: 674%

Hectáas mínimas rentables: 14,8 ha

 Es la inversión más eficiente.

Muy bajo coste (2.500 €)

Alta rentabilidad relativa

Se amortiza incluso en explotaciones pequeñas.


RTK incremental

VAN (15 años): 168.037 €

TIR: 241%

Hectáreas mínimas: 41,5 ha

 Muy rentable, especialmente en explotaciones medianas-grandes.

Corte por tramos (máquina fitosanitarios)

VAN (15 años): 335.459 € (la mayor en términos absolutos)

TIR: ~61%

Hectáreas mínimas: 161,9 ha

 Es la que más dinero genera en explotaciones grandes.

Siembra variable

VAN (15 años): 157.839 €

TIR: ~53%

Hectáreas mínimas: 188 ha

 Rentable, pero requiere tamaño alto.
Es inversión estructural, no para pequeñas explotaciones.


Arroz
Siembra variable

VAN (15 años): 135.083 €

TIR: ~46%

Hectáreas mínimas: 213 ha

 Es la tecnología clave en arroz.
La mejora viene principalmente del rendimiento.

ISOBUS

VAN (15 años): 58.915 €

TIR: 236%

Hectáreas mínimas: 42 ha

 Muy eficiente en términos relativos (como en maíz).

RTK incremental

VAN (15 años): 54.415 €

TIR: 84%

Hectáreas mínimas: 118 ha

Rentable en explotaciones medianas.

Corte por tramos

VAN (15 años): 109.614 €

TIR: 25%

Hectáreas mínimas: 375 ha

 Rentable solo en explotaciones muy grandes.

Abono variable

VAN positivo

TIR ~21%

Necesita 446 ha

Rentable pero requiere tamaño elevado.

In [ ]:
#tabla resumen final
def resumen_final(tabla_maiz_vida, tabla_arroz_vida, vida=15, superficie=100):

    def preparar(df, cultivo):
        df2 = df.copy()
        df2["Cultivo"] = cultivo
        
        df2["VAN_final"] = df2[f"VAN_{superficie}ha_{vida}a_€"]
        df2["TIR_final_%"] = df2[f"TIR_{superficie}ha_{vida}a_%"]
        
        # Clasificación sencilla
        df2["Clasificacion"] = np.where(
            df2["VAN_final"] > 0,
            np.where(df2["TIR_final_%"] > 50, "Alta rentabilidad", "Rentabilidad media"),
            "No rentable"
        )
        
        return df2[[
            "Cultivo",
            "Tecnologia",
            "Efecto_€/ha",
            "Coste_eur",
            "Hectareas_minimas_rentables",
            "VAN_final",
            "TIR_final_%",
            "Clasificacion"
        ]]

    resumen = pd.concat([
        preparar(tabla_maiz_vida, "Maíz"),
        preparar(tabla_arroz_vida, "Arroz")
    ])

    return resumen.sort_values(["Cultivo", "VAN_final"], ascending=[True, False])


tabla_resumen_final = resumen_final(tabla_maiz_vida, tabla_arroz_vida, vida=15, superficie=100)

print("=== TABLA RESUMEN FINAL (100 ha, 15 años, r=5%) ===")
print(tabla_resumen_final.round(2))


In [ ]:

def grafico_van(tabla_resumen, cultivo):
    df = tabla_resumen[tabla_resumen["Cultivo"] == cultivo].copy()
    df = df[df["Tecnologia"] != "GPS"]
    df = df.sort_values("VAN_final", ascending=True)

    plt.figure()
    plt.barh(df["Tecnologia"], df["VAN_final"])
    plt.axvline(0, linestyle="--")
    plt.xlabel("VAN (€) a 15 años (100 ha)")
    plt.title(f"{cultivo} - VAN por tecnología")
    plt.tight_layout()
    #plt.savefig("VAN_cultivo.png", dpi=300, bbox_inches="tight")
    plt.show()



In [ ]:

grafico_van(tabla_resumen_final, "Maíz")


In [ ]:
grafico_van(tabla_resumen_final, "Arroz")


In [ ]:
#hectáreas mínimas necesarias

def grafico_hectareas(tabla_resumen, cultivo, lineas_ref=(50,100,200)):
    df = tabla_resumen[
        (tabla_resumen["Cultivo"] == cultivo) &
        (tabla_resumen["Hectareas_minimas_rentables"].notnull())].copy()

    df = df.sort_values("Hectareas_minimas_rentables", ascending=True)

    plt.figure()
    plt.barh(df["Tecnologia"], df["Hectareas_minimas_rentables"])
    plt.xlabel("Hectáreas mínimas (en una campaña) para recuperar la inversión")
    plt.title(f"{cultivo} - Umbral mínimo de rentabilidad")

    if lineas_ref:
        for h in lineas_ref:
            plt.axvline(h, linestyle="--", linewidth=1)
            plt.text(h, -0.5, f"{h} ha", rotation=90, va="bottom", ha="right")

    
    plt.tight_layout()
    plt.savefig("Umbral_superficie_rentabilidad.png", dpi=300, bbox_inches="tight")
    plt.show()




In [ ]:
grafico_hectareas(tabla_resumen_final, "Maíz", lineas_ref=(50,100,200))

In [ ]:
grafico_hectareas(tabla_resumen_final, "Arroz", lineas_ref=(50,100,200))


In [ ]:

def grafico_estrategico(tabla, cultivo):

    df = tabla[
        (tabla["Cultivo"] == cultivo) &
        (tabla["Hectareas_minimas_rentables"].notnull())
    ].copy()

    plt.figure()
    plt.scatter(
        df["Hectareas_minimas_rentables"],
        df["VAN_final"]
    )

    for i in range(len(df)):
        plt.text(
            df["Hectareas_minimas_rentables"].iloc[i],
            df["VAN_final"].iloc[i],
            df["Tecnologia"].iloc[i]
        )

    plt.axhline(0)
    plt.xlabel("Hectáreas mínimas necesarias")
    plt.ylabel("VAN (15 años, 100 ha)")
    plt.title(f"{cultivo} - Mapa estratégico tecnologías")
    plt.tight_layout()
    plt.show()


grafico_estrategico(tabla_resumen_final, "Maíz")
grafico_estrategico(tabla_resumen_final, "Arroz")


In [ ]:
# MAÍZ - TIR

maiz_tir_plot = maiz_van[maiz_van["Tecnologia"] != "GPS"]
maiz_tir_plot = maiz_tir_plot.sort_values("TIR_100ha_%", ascending=False)

plt.figure()
plt.bar(maiz_tir_plot["Tecnologia"], maiz_tir_plot["TIR_100ha_%"])

plt.xticks(rotation=45)
plt.title("MAÍZ - TIR (100 ha)")
plt.ylabel("TIR (%)")

plt.axhline(0, linestyle="--")

plt.tight_layout()
#plt.savefig("TIR_maiz.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ARROZ - TIR

arroz_tir_plot = arroz_van[arroz_van["Tecnologia"] != "GPS"]
arroz_tir_plot = arroz_tir_plot.sort_values("TIR_100ha_%", ascending=False)

plt.figure()
plt.bar(arroz_tir_plot["Tecnologia"], arroz_tir_plot["TIR_100ha_%"])

plt.xticks(rotation=45)
plt.title("ARROZ - TIR (100 ha)")
plt.ylabel("TIR (%)")


plt.tight_layout()
#plt.savefig("TIR_arroz.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# MAÍZ - PAYBACK

maiz_payback_plot = maiz_van[maiz_van["Tecnologia"] != "GPS"]
maiz_payback_plot = maiz_payback_plot.sort_values("Payback_100ha_años")

plt.figure()
plt.bar(maiz_payback_plot["Tecnologia"], maiz_payback_plot["Payback_100ha_años"])

plt.xticks(rotation=45)
plt.title("MAÍZ - Payback (100 ha)")
plt.ylabel("Años para recuperar la inversión")


plt.tight_layout()
plt.savefig("Años_recuperar_inversion_maiz.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ARROZ - PAYBACK

arroz_payback_plot = arroz_van[arroz_van["Tecnologia"] != "GPS"]
arroz_payback_plot = arroz_payback_plot.sort_values("Payback_100ha_años")

plt.figure()
plt.bar(arroz_payback_plot["Tecnologia"], arroz_payback_plot["Payback_100ha_años"])

plt.xticks(rotation=45)
plt.title("ARROZ - Payback (100 ha)")
plt.ylabel("Años para recuperar la inversión")

plt.tight_layout()
plt.savefig("Años_recuperar_inversion_arroz.png", dpi=300, bbox_inches="tight")

plt.show()